# Data Quality & Retail Exploration

**Pregunta:** ¿qué ventas puedo analizar con confianza?

Fuente real: UCI Online Retail II, 2009-2011, moneda GBP. Las reglas y exclusiones se muestran antes de interpretar los gráficos. Ejecuta las celdas en orden; el primer acceso descarga la fuente pública y luego reutiliza una copia verificada.

## 1. Preparación

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of retail-analytics-pipeline.")
sys.path.insert(0, str(ROOT / "src"))
REPORTS = ROOT / "reports"


**Dónde:** Jupyter en Codespaces o una copia del repositorio. La carpeta raíz se identifica por `pyproject.toml`; los mismos módulos se usan desde `main.py`.

## 2. Carga de la fuente

In [2]:
from market_insights.data import load_transactions
raw, source = load_transactions(ROOT / 'data/raw')
display(pd.DataFrame([source]).drop(columns=['cache_sha256'], errors='ignore'))
print('Filas originales:', len(raw))
display(raw.head())

Filas originales: 1067371


,source,source_url,license,currency,zip_sha256,raw_rows,sheets
0,UCI Online Retail II,https://archive.ics.uci.edu/dataset/502/online...,CC BY 4.0,GBP,572e36277c2390fbfde10664750731e0a86f55e33470d9...,1067371,"[Year 2009-2010, Year 2010-2011]"


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Stack:** pandas, openpyxl y urllib. Se leen ambas hojas del archivo público. La descarga y el CSV de caché se identifican mediante SHA-256. Las fechas y la moneda se conservan; no se convierten en datos LATAM.

## 3. Auditoría y contrato

In [3]:
from market_insights.quality import clean_transactions
sales, audit = clean_transactions(raw)
display(pd.DataFrame(list(audit.items()), columns=['regla','resultado']))
display(sales[['order_id','product_id','order_date','quantity','unit_price','country']].head())

,regla,resultado
0,input_rows,1067371
1,exact_duplicate_rows,34335
2,invalid_required_rows,0
3,cancelled_or_nonpositive_quantity_rows,22497
4,nonpositive_price_rows,2626
5,nonproduct_code_rows,4699
6,retained_rows_without_customer_id,226637
7,retained_rows,1003214
8,date_min,2009-12-01 07:45:00
9,date_max,2011-12-09 12:50:00


,order_id,product_id,order_date,quantity,unit_price,country
0,489434,85048,2009-12-01 07:45:00,12,6.95,United Kingdom
1,489434,79323P,2009-12-01 07:45:00,12,6.75,United Kingdom
2,489434,79323W,2009-12-01 07:45:00,12,6.75,United Kingdom
3,489434,22041,2009-12-01 07:45:00,48,2.10,United Kingdom
4,489434,21232,2009-12-01 07:45:00,24,1.25,United Kingdom


**Stack:** pandas. Se normalizan columnas antes de validarlas. Se contabilizan duplicados exactos, cancelaciones, cantidades no positivas, precios no positivos y códigos que no son productos. Los clientes ausentes se conservan para análisis agregado. Esta tabla representa ventas pagadas positivas; no ingresos netos después de devoluciones.

## 4. SQL y calendario

In [4]:
from market_insights.queries import export_queries
from market_insights.visualization import plot_sql
sql_results = export_queries(sales, ROOT/'data/processed/retail.duckdb', ROOT/'sql', REPORTS/'metrics')
display(sql_results['03_orders_by_dow'])
display(sql_results['01_top_products'])
plot_sql(sql_results, 'UCI Online Retail II | retained paid product sales', REPORTS/'figures')

,order_dow,day_name,total_orders
0,0,Mon,6245
1,1,Tue,7147
2,2,Wed,7115
3,3,Thu,8167
4,4,Fri,5988
5,5,Sat,30
6,6,Sun,4824


,product_id,product_name,units_sold,orders_count
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,106139.0,1019
1,85099B,RED RETROSPOT JUMBO BAG,96757.0,3989
2,21212,PACK OF 72 RETROSPOT CAKE CASES,94884.0,3118
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,94203.0,5365
4,22197,SMALL POPCORN HOLDER,88499.0,2322
5,23843,"PAPER CRAFT , LITTLE BIRDIE",80995.0,1
6,84879,ASSORTED COLOUR BIRD ORNAMENT,80082.0,2807
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033.0,247
8,17003,BROCADE RING PURSE,70369.0,456
9,21977,PACK OF 60 PINK PAISLEY CAKE CASES,56061.0,1993


**Stack:** DuckDB, SQL y Matplotlib. Lunes es 0 porque se deriva desde la fecha. Las columnas del heatmap corresponden a horas reales, de 00:00 a 23:00. El recuento es de pedidos distintos.

![Figura de la ejecución registrada](../reports/figures/orders_heatmap.png)

## 5. Preparar la serie de forecasting

In [5]:
from market_insights.data import real_weekly
from market_insights.quality import validate_weekly
weekly, cutoff, scope = real_weekly(sales)
weekly.to_csv(ROOT/'data/processed/weekly_real.csv', index=False)
display(pd.DataFrame([validate_weekly(weekly)]))
print('Inicio del holdout:', cutoff.date())
print('SKU de ejemplo seleccionado con train:', scope['example_series'])
from market_insights.operations import daily_orders
daily, daily_scope = daily_orders(sales)
daily.to_csv(ROOT/"data/processed/daily_orders.csv", index=False)
display(pd.DataFrame([daily_scope]))


Inicio del holdout: 2011-08-15
SKU de ejemplo seleccionado con train: 85123A


,rows,series,weeks,missing_by_column
0,1248,12,104,"{'segment': 0, 'week_start': 0, 'units_sold': ..."


,target,country,source,covered_first_day,covered_last_day,calendar_days,zero_policy,horizon
0,Distinct retained paid UK order IDs within eac...,United Kingdom,UCI Online Retail II,2009-12-07,2011-12-04,728,No retained order recorded in this covered day...,Next day; observations update after each compl...


**Stack:** pandas. El pronóstico trabaja con 12 SKU del Reino Unido, elegidos por volumen exclusivamente del entrenamiento y con actividad suficiente. Se excluyen semanas inicial/final incompletas. Un cero significa ninguna venta pagada registrada en una semana cubierta; no demuestra demanda cero ni disponibilidad de inventario.

## 6. Lectura de negocio

Distinguir calidad de datos y condiciones reales del negocio. Las devoluciones no desaparecen del fenómeno económico porque este análisis las excluya; están fuera de su target. Una venta inusual puede ser legítima. Consultar los errores del modelo antes de proponer reglas automáticas de outliers.